# Kaggle Global Airports Dataset — Cleaning

Cleans the [Kaggle Global Airports Dataset](https://www.kaggle.com/datasets/samvelkoch/global-airports-iata-icao-timezone-geo) and writes one cleaned Kaggle-side dataset to `data/interim/`. This source's main added value is the **TimeZone** attribute, which the other two sources lack.

Steps: strip airport-name whitespace, drop duplicated-ICAO non-airport/stale rows, null out "Null Island" `(0, 0)` coordinates, sanity-check coordinate bounds, and export.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent

RAW_DIR = project_root / "data" / "raw" / "kaggle"
INTERIM_DIR = project_root / "data" / "interim"

data = pd.read_csv(RAW_DIR / "airports-kaggle.csv")
print(f"raw Kaggle airports: {data.shape}")

raw Kaggle airports: (6393, 13)


## Overview

In [2]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6393 entries, 0 to 6392
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   AirportName         6393 non-null   object 
 1   IATA                6373 non-null   object 
 2   ICAO                5091 non-null   object 
 3   TimeZone            6305 non-null   object 
 4   City_Name           6393 non-null   object 
 5   City_IATA           6331 non-null   object 
 6   UTC_Offset_Hours    6305 non-null   float64
 7   UTC_Offset_Seconds  6305 non-null   float64
 8   Country_CodeA2      6379 non-null   object 
 9   Country_CodeA3      6393 non-null   object 
 10  Country_Name        6393 non-null   object 
 11  GeoPointLat         6393 non-null   float64
 12  GeoPointLong        6393 non-null   float64
dtypes: float64(4), object(9)
memory usage: 649.4+ KB


In [3]:
data.head()

,AirportName,IATA,ICAO,TimeZone,City_Name,City_IATA,UTC_Offset_Hours,UTC_Offset_Seconds,Country_CodeA2,Country_CodeA3,Country_Name,GeoPointLat,GeoPointLong
0,Pilanesberg Intl,NTY,FAPN,Africa/Johannesburg,Sun City,NTY,2.0,7200.0,ZA,ZAF,South Africa,-25.333822,27.173358
1,Clovis Muni,CVN,KCVN,America/Denver,"Clovis, New Mexico",CVN,-6.0,-21600.0,US,USA,United States of America,34.425139,-103.079278
2,Cannon Afb,CVS,KCVS,America/Denver,"Clovis, New Mexico",CVN,-6.0,-21600.0,US,USA,United States of America,34.382775,-103.322147
3,Scammon Bay Airport,SCM,PACM,America/Nome,"Scammon Bay, Alaska",SCM,-8.0,-28800.0,US,USA,United States of America,61.845278,-165.571389
4,Kapit,KPI,NaN,Asia/Kuching,Kapit,KPI,8.0,28800.0,MY,MYS,Malaysia,2.017000,112.950000


## Missingness

In [4]:
(data.isna().mean() * 100).round(2).rename("missing_%")

AirportName            0.00
IATA                   0.31
ICAO                  20.37
TimeZone               1.38
City_Name              0.00
City_IATA              0.97
UTC_Offset_Hours       1.38
UTC_Offset_Seconds     1.38
Country_CodeA2         0.22
Country_CodeA3         0.00
Country_Name           0.00
GeoPointLat            0.00
GeoPointLong           0.00
Name: missing_%, dtype: float64

## Pre-processing

- strip leading/trailing whitespace from `AirportName`
- drop rows whose ICAO code is duplicated but doesn't belong to the airport (a stale/transferred code, or a non-airport record)

In [5]:
# Strip whitespace around airport names (several entries have padding).
data["AirportName"] = data["AirportName"].str.strip()

### Duplicated ICAO codes

ICAO should uniquely identify one airport, so it serves as the gold-standard set for evaluation — any duplicate is a data-quality issue to resolve.

In [6]:
icao_present = data[data["ICAO"].notna()]
icao_dup = icao_present["ICAO"].duplicated(keep=False)

print("Duplicated ICAO values:", icao_present.loc[icao_dup, "ICAO"].nunique())
print("Rows involved:", int(icao_dup.sum()))
icao_present.loc[icao_dup, ["AirportName", "IATA", "ICAO", "City_Name", "Country_Name"]].sort_values("ICAO")

Duplicated ICAO values: 2
Rows involved: 4


,AirportName,IATA,ICAO,City_Name,Country_Name
5766,Schonefeld,SXF,EDDB,Berlin,Germany
5767,Berlin Brandenburg Willy Brandt,BER,EDDB,Berlin,Germany
4096,Dortmund,DTM,EDLW,Dortmund,Germany
4097,HBF RAILWAY STATION,DTZ,EDLW,Dortmund,Germany


In [7]:
# EDDB: the code was transferred from the (closed) Berlin Schönefeld to the new
#       Berlin Brandenburg Airport (BER) -> keep the current airport, drop "Schonefeld".
# EDLW: "HBF RAILWAY STATION" is a train station, not an airport, and should not carry an
#       ICAO code -> drop it, keep the Dortmund airport record.
drop_mask = ((data["ICAO"] == "EDDB") & (data["AirportName"] == "Schonefeld")) | \
            ((data["ICAO"] == "EDLW") & (data["AirportName"] == "HBF RAILWAY STATION"))

print("Rows removed:", int(drop_mask.sum()))
data = data[~drop_mask].reset_index(drop=True)
print("Remaining rows:", data.shape[0])

Rows removed: 2
Remaining rows: 6391


## Geographic validation

Coordinates are central to identity resolution and fusion, so they're checked here. `(0, 0)` ("Null Island") is not a real airport location — these are missing values encoded as zeros, so we set them to `NaN` instead of letting them vote with a bogus coordinate later. They can be recovered from the other two sources during integration.

In [8]:
null_island = (data["GeoPointLat"] == 0) & (data["GeoPointLong"] == 0)
print(f"Null-Island (0, 0) rows set to NaN: {int(null_island.sum())}")
data.loc[null_island, ["GeoPointLat", "GeoPointLong"]] = np.nan

out_of_bounds = (data["GeoPointLat"].abs() > 90) | (data["GeoPointLong"].abs() > 180)
print(f"out-of-bounds coordinates remaining: {int(out_of_bounds.sum())}")

Null-Island (0, 0) rows set to NaN: 319
out-of-bounds coordinates remaining: 0


## Export

In [9]:
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
out_path = INTERIM_DIR / "kaggle_airports.csv"
data.to_csv(out_path, index=False)
print(f"Saved {len(data):,} cleaned Kaggle airports to {out_path}")

Saved 6,391 cleaned Kaggle airports to /Users/I569494/Development/Uni Mannheim/Web-Data-Integration-Project/data/interim/kaggle_airports.csv


## Notes

- Native Kaggle column names are kept (not renamed to the integrated schema) — same convention as the other sources' interim files; the mapping to the target schema lives in `schema/integrated_schema.xsd`.
- `(0, 0)` coordinates (319 rows) are nulled, not dropped: the rest of each row (name, codes, timezone) is still useful, and the coordinate can be filled from OurAirports/Wikidata during fusion.